# Part 5 - Design Documents and Test Cases

## Goal

From the planning document:

- *"Functional and Technical Design Agents produce the required documents, running in parallel when both are needed."*
- *"A Test Agent generates functional, technical, integration, and regression cases."*
- *"A document tool exports designs."*
- The coordination uses *"sequential handoffs and one parallel design branch"*, because the functional and
  technical designs *"can be produced independently from the same specification"*.

This notebook builds that branch with **LangGraph**:

**approved package -> (Functional Design Agent || Technical Design Agent) -> Test Agent -> Word documents**

It also adds the **Change Request (CR)** document: the **CR Assessment Agent** decides from the company CR
standard (`STD-CR-01`) whether a request needs a CR at all, and the **Change Request Agent** fills the company's
*ERP Change/Additional Request Form*, exported to Word for the customer to sign. Finally, every document is filed on **SharePoint** as
`<root>/<Customer>/<ID - Description>/<Document type>/`.

## Setup

Same environment as the course notebooks: a local **Ollama** server running **`gemma4:e4b`** for the agents and **`nomic-embed-text`** for embeddings. The cells work on Google Colab (Ollama is installed automatically) and locally (install Ollama from ollama.com first).

This notebook only needs the chat model.

**Chat model:** set `LLM_PROVIDER=openai` in `.env` to run the agents on **GPT Luna** (`OPENAI_MODEL`, default `gpt-6-luna`, key in `OPEN_AI_KEY`); Ollama is then not needed for this notebook. `LLM_PROVIDER=ollama` (the default) keeps the local `gemma4:e4b`.

In [ ]:
%pip install -q -U langchain langchain-core langchain-ollama langchain-openai python-dotenv langchain-text-splitters langgraph langchain-mcp-adapters mcp python-docx requests numpy langsmith

In [ ]:
import os
import platform
import shutil
import subprocess
import time

import requests
from dotenv import load_dotenv

load_dotenv()  # reads .env next to the notebook (OPEN_AI_KEY, LLM_PROVIDER, ADO_*, ...) when it exists

# Chat model: "openai" runs the agents on GPT Luna (key in OPEN_AI_KEY), "ollama" on the local model.
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "ollama").strip().lower()
if LLM_PROVIDER not in ("openai", "ollama"):
    raise ValueError(f"LLM_PROVIDER must be 'openai' or 'ollama', not {LLM_PROVIDER!r}")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma4:e4b")
LLM_MODEL = OPENAI_MODEL if LLM_PROVIDER == "openai" else OLLAMA_MODEL
EMBED_MODEL = os.getenv("EMBED_MODEL", "nomic-embed-text")
OLLAMA_URL = "http://localhost:11434"
NEEDS_OLLAMA = LLM_PROVIDER == "ollama"  # this notebook only needs the chat model


def ollama_is_running() -> bool:
    try:
        return requests.get(f"{OLLAMA_URL}/api/tags", timeout=3).ok
    except requests.RequestException:
        return False


# 1. Install Ollama (Colab / Linux) and start the server in the background
if NEEDS_OLLAMA:
    if shutil.which("ollama") is None:
        if platform.system() == "Linux":
            print("Installing Ollama...")
            !sudo apt-get install -y -qq zstd
            !curl -fsSL https://ollama.com/install.sh | sh
        else:
            print("Install Ollama from https://ollama.com/download, then re-run this cell.")

    if not ollama_is_running():
        print("Starting Ollama server...")
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(5)  # Wait for server to wake up

    print("Ollama running:", ollama_is_running())
print(f"Chat model: {LLM_PROVIDER} / {LLM_MODEL}")

In [ ]:
# 2. Pull the chat model and the embedding model (only when the agents run on Ollama)
if NEEDS_OLLAMA:
    !ollama pull {LLM_MODEL}
    !ollama pull {EMBED_MODEL}
    !ollama list

### LangSmith tracing

Every agent, LLM call, MCP tool call and LangGraph node is traced in **LangSmith**, so you can open each step
and see its **prompt, output, duration and tokens**. Paste your API key when asked (or store it as the Colab
secret `LANGSMITH_API_KEY`); press Enter to run without tracing.

In [ ]:
import getpass
import os

import langsmith.utils

# LangSmith records every agent, LLM call (full prompt and answer), MCP tool call and LangGraph node,
# with its duration and token usage. Free API key: https://smith.langchain.com -> Settings -> API Keys.
# Note: traces contain the prompts, so customer emails are sent to LangSmith. Use test data or your own workspace.
USE_LANGSMITH = True
LANGSMITH_PROJECT = "fo-requirements-to-delivery"


def read_langsmith_key() -> str:
    if os.getenv("LANGSMITH_API_KEY"):
        return os.environ["LANGSMITH_API_KEY"]
    try:
        from google.colab import userdata  # Colab: add a secret named LANGSMITH_API_KEY

        return userdata.get("LANGSMITH_API_KEY")
    except Exception:
        pass
    try:
        return getpass.getpass("LangSmith API key (press Enter to run without tracing): ").strip()
    except Exception:  # no interactive input available
        return ""


api_key = read_langsmith_key() if USE_LANGSMITH else ""
if api_key:
    os.environ.update(LANGSMITH_API_KEY=api_key, LANGSMITH_TRACING="true", LANGSMITH_PROJECT=LANGSMITH_PROJECT)
    print(f"LangSmith tracing ON -> project '{LANGSMITH_PROJECT}' on https://smith.langchain.com")
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing OFF")
langsmith.utils.get_env_var.cache_clear()  # LangSmith caches environment variables

In [ ]:
import getpass
import os


def make_llm():
    """The chat model selected by LLM_PROVIDER: GPT Luna on OpenAI, or the local Ollama model."""
    if LLM_PROVIDER == "openai":
        from langchain_openai import ChatOpenAI

        if not (os.getenv("OPEN_AI_KEY") or os.getenv("OPENAI_API_KEY")):
            os.environ["OPEN_AI_KEY"] = getpass.getpass("OpenAI API key: ").strip()
        # GPT Luna only accepts its default temperature, so none is set.
        return ChatOpenAI(model=LLM_MODEL, api_key=os.getenv("OPEN_AI_KEY") or os.getenv("OPENAI_API_KEY"))

    from langchain_ollama import ChatOllama

    return ChatOllama(model=LLM_MODEL, temperature=0)


llm = make_llm()

In [ ]:
import json

from langchain_core.messages import HumanMessage, SystemMessage
from langsmith import traceable  # names agent steps in LangSmith (does nothing when tracing is off)


def ask_structured(schema, system_prompt: str, user_prompt: str, retries: int = 2):
    """Call the LLM and force the answer into a Pydantic schema. On failure, retry with the error message."""
    # The run name shows up in LangSmith, e.g. "DeliveryPlan" or "GapAnalysis".
    structured_llm = llm.with_structured_output(schema, method="json_schema").with_config(run_name=schema.__name__)
    last_error = None

    for attempt in range(retries + 1):
        try:
            result = structured_llm.invoke(
                [SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]
            )
            if result is None:
                raise ValueError("The model returned an empty answer.")
            return result
        except Exception as error:
            last_error = error
            print(f"[retry {attempt + 1}] invalid structured output: {error}")
            user_prompt += (
                f"\n\nYour previous answer was invalid ({error}). "
                "Return JSON that matches the schema exactly."
            )

    raise last_error


def ask_text(system_prompt: str, user_prompt: str) -> str:
    return llm.invoke(
        [SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]
    ).text  # plain string for both providers (OpenAI returns content blocks)


def show(title: str, obj) -> None:
    """Pretty-print a heading and a Pydantic object, dict, list or string."""
    print(f"\n========== {title} ==========\n")
    if hasattr(obj, "model_dump"):
        obj = obj.model_dump()
    print(obj if isinstance(obj, str) else json.dumps(obj, indent=2, ensure_ascii=False))
import operator
import time
from pathlib import Path
from typing import Annotated, TypedDict

from langgraph.graph import StateGraph, START, END

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class Requirement(BaseModel):
    req_id: str = Field(description="Stable ID such as REQ-001")
    statement: str = Field(description="One clear, testable requirement sentence")
    source_text: str = Field(description="The customer's words this requirement comes from")
    actor: str = Field(description="Who performs or benefits from the requirement")
    business_rules: list[str] = Field(description="Rules, thresholds or conditions that are actually stated")
    exceptions: list[str] = Field(description="Exceptions or edge cases that are actually stated")
    expected_outcome: str = Field(description="What should happen when the requirement is met")


class RequirementExtraction(BaseModel):
    summary: str = Field(description="Two-sentence summary of the request")
    actors: list[str]
    in_scope: list[str]
    out_of_scope: list[str]
    security_constraints: list[str] = Field(description="Security, approval or segregation-of-duties constraints")
    requirements: list[Requirement]


class Classification(BaseModel):
    module: Literal["Finance", "Supply Chain", "Finance and Supply Chain", "Out of scope"]
    sub_areas: list[str] = Field(description="F&O areas such as Accounts payable, Procurement, Inventory")
    work_type: Literal["Functional", "Technical", "Mixed"]
    design_documents: Literal["FDD", "TDD", "FDD and TDD"]
    risk_level: Literal["Low", "Medium", "High"]
    rationale: str


class Gap(BaseModel):
    gap_id: str = Field(description="Stable ID such as GAP-01")
    req_id: str = Field(description="The requirement this gap belongs to")
    missing_information: str
    why_it_matters: str
    clarification_question: str = Field(description="One clear question the customer can answer")
    assumption_if_unanswered: str


class GapAnalysis(BaseModel):
    gaps: list[Gap]
class AcceptanceCriterion(BaseModel):
    ac_id: str = Field(description="ID such as AC-01")
    req_ids: list[str]
    given: str
    when: str
    then: str


class TaskItem(BaseModel):
    task_id: str = Field(description="ID such as T-01")
    kind: Literal["Functional", "Technical"]
    title: str = Field(description="Starts with FUNC: or TECH:")
    description: str
    req_ids: list[str]
    estimate_hours: float


class UserStory(BaseModel):
    story_id: str = Field(description="ID such as US-01")
    title: str = Field(description="Format: [Module] short goal")
    as_a: str
    i_want: str
    so_that: str
    req_ids: list[str]
    acceptance_criteria: list[AcceptanceCriterion]
    tasks: list[TaskItem]


class DeliveryPlan(BaseModel):
    user_stories: list[UserStory]
class DesignSection(BaseModel):
    section_id: str = Field(description="ID such as FDD-1 or TDD-1")
    heading: str
    content: str
    req_ids: list[str] = Field(description="Requirement IDs this section covers")


class DesignDocument(BaseModel):
    doc_type: Literal["FDD", "TDD"]
    title: str
    overview: str
    sections: list[DesignSection]
    open_points: list[str]


class TestCase(BaseModel):
    test_id: str = Field(description="ID such as TC-01")
    test_type: Literal["Functional", "Technical", "Integration", "Regression"]
    title: str
    req_ids: list[str]
    preconditions: str
    steps: list[str]
    expected_result: str


class TestSuite(BaseModel):
    test_cases: list[TestCase]

## Inputs: Approved Specification and Delivery Plan

The design agents start from the approved package (immutable facts, requirements, evidence) and the plan from
notebook 04. Both are included so this notebook can run on its own.

In [ ]:
REQUEST_ID = "CR-2026-001"

CUSTOMER_REQUEST = """
Subject: Vendor invoices approval + blocking vendors

Hi team,

Going forward we need vendor invoices above a certain amount to be approved by the
finance manager before they can be posted. Right now anyone in AP can post them and
the auditors flagged it.

Also, vendors that don't have bank details should be blocked so we don't pay them by
mistake. Some of the old vendors are like this too.

Can we have this for all our companies? We would like it live before the quarter close.

Thanks,
Rana Haddad
AP Team Lead, Cedar Retail Group
"""
REQUEST_ID = "CR-2026-001"
DEADLINE = "2026-12-15"

CLASSIFICATION = {
    "module": "Finance",
    "sub_areas": ["Accounts payable"],
    "work_type": "Mixed",
    "design_documents": "FDD and TDD",
    "risk_level": "Medium",
    "rationale": "Approval workflow configuration (functional) plus a vendor payment-hold batch job and validation (technical).",
}

APPROVED_REQUIREMENTS = [
    {"req_id": "REQ-001", "actor": "Finance Manager",
     "statement": "Vendor invoices with a total including tax greater than 10,000 USD (accounting currency) must be approved by the Finance Manager of the legal entity before they can be posted.",
     "business_rules": ["Threshold 10,000 USD incl. tax, accounting currency", "Legal entities USMF and DEMF"],
     "exceptions": [], "expected_outcome": "Invoices above the threshold cannot be posted until approved."},
    {"req_id": "REQ-002", "actor": "AP Clerk",
     "statement": "Vendor invoices of 10,000 USD or less can be posted by AP clerks without approval.",
     "business_rules": ["Threshold 10,000 USD incl. tax"], "exceptions": [],
     "expected_outcome": "Current posting process is unchanged for small invoices."},
    {"req_id": "REQ-003", "actor": "CFO",
     "statement": "If the Finance Manager does not act within 2 business days, the approval escalates to the CFO.",
     "business_rules": ["Escalation after 2 business days"], "exceptions": [],
     "expected_outcome": "Pending approvals do not block posting indefinitely."},
    {"req_id": "REQ-004", "actor": "AP Team",
     "statement": "Existing and new vendors without at least one active bank account are placed on payment hold, which blocks payment generation but still allows invoice registration.",
     "business_rules": ["On hold = Payment", "Applies to existing and new vendors"], "exceptions": [],
     "expected_outcome": "No payment can be generated for vendors without bank details."},
    {"req_id": "REQ-005", "actor": "Auditor",
     "statement": "The approval history (approver, decision, date and time) of each vendor invoice must be visible to auditors.",
     "business_rules": [], "exceptions": [], "expected_outcome": "Auditors can prove who approved each invoice and when."},
    {"req_id": "REQ-006", "actor": "AP Clerk",
     "statement": "AP clerks must not be able to approve vendor invoices they entered themselves.",
     "business_rules": ["Segregation of duties"], "exceptions": [],
     "expected_outcome": "Self-approval is technically prevented."},
]

LOCKED_FACTS = [
    "Request ID: CR-2026-001",
    "Legal entities in scope: USMF and DEMF only",
    "Approval threshold: vendor invoice total including tax greater than 10,000 USD in accounting currency",
    "Approver: Finance Manager of the legal entity; escalate to the CFO after 2 business days",
    "Blocked vendor means On hold = Payment (invoices can still be registered)",
    "Security: AP clerks must not approve invoices they entered",
    "Go-live deadline: 2026-12-15",
    "Cited evidence: MS-WF-01, MS-AP-01, MS-SEC-01, DES-2025-014, DES-2025-031",
]

EVIDENCE = [
    {"doc_id": "MS-WF-01", "title": "Workflow and approvals", "source_type": "microsoft_guidance", "module": "Finance",
     "snippet": "Approval steps can have conditions, such as the invoice total being greater than an amount... time limits and escalation paths... workflow history records who approved and when."},
    {"doc_id": "MS-AP-01", "title": "Vendor invoices and holds", "source_type": "microsoft_guidance", "module": "Finance",
     "snippet": "On hold = Payment blocks payment generation while invoices can still be registered."},
    {"doc_id": "DES-2025-014", "title": "Approved design: purchase invoice approval by amount", "source_type": "approved_design", "module": "Finance",
     "snippet": "Workflow condition on invoice total (accounting currency) > threshold; SoD rule; threshold in a CUS parameter."},
    {"doc_id": "DES-2025-031", "title": "Approved design: automatic hold for vendors with incomplete master data", "source_type": "approved_design", "module": "Finance",
     "snippet": "Recurring SysOperation batch sets On hold = Payment; CoC validation on vendor insert/update; CUS log table."},
]
# Output of the Planning Agent for CR-2026-001 (see notebook 04)
SAMPLE_PLAN = {"user_stories": [
    {"story_id": "US-01", "title": "[Finance] Approve large vendor invoices before posting",
     "as_a": "Finance Manager", "i_want": "to approve vendor invoices above 10,000 USD before they are posted",
     "so_that": "large payments are controlled and auditable",
     "req_ids": ["REQ-001", "REQ-002", "REQ-003", "REQ-005", "REQ-006"],
     "acceptance_criteria": [
         {"ac_id": "AC-01", "req_ids": ["REQ-001"], "given": "a vendor invoice in USMF with a total of 10,000.01 USD including tax",
          "when": "an AP clerk submits it", "then": "it cannot be posted until the Finance Manager approves it"},
         {"ac_id": "AC-02", "req_ids": ["REQ-002"], "given": "a vendor invoice of 10,000 USD or less",
          "when": "an AP clerk posts it", "then": "it posts without approval"},
         {"ac_id": "AC-03", "req_ids": ["REQ-003"], "given": "an approval pending for more than 2 business days",
          "when": "the escalation time limit is reached", "then": "the approval is escalated to the CFO"},
         {"ac_id": "AC-04", "req_ids": ["REQ-005"], "given": "an approved vendor invoice",
          "when": "an auditor opens the workflow history", "then": "the approver, decision, date and time are shown"},
         {"ac_id": "AC-05", "req_ids": ["REQ-006"], "given": "an AP clerk who entered an invoice",
          "when": "the same clerk tries to approve it", "then": "the approval is prevented"},
     ],
     "tasks": [
         {"task_id": "T-01", "kind": "Functional", "title": "FUNC: Configure the vendor invoice workflow with threshold and escalation in USMF and DEMF",
          "description": "Workflow condition on invoice total incl. tax > 10,000 USD, Finance Manager step, CFO escalation after 2 business days.",
          "req_ids": ["REQ-001", "REQ-002", "REQ-003"], "estimate_hours": 12},
         {"task_id": "T-02", "kind": "Functional", "title": "FUNC: Write the FDD for invoice approval and vendor payment hold",
          "description": "Functional design document.", "req_ids": ["REQ-001", "REQ-002", "REQ-003", "REQ-004", "REQ-005", "REQ-006"], "estimate_hours": 8},
         {"task_id": "T-03", "kind": "Technical", "title": "TECH: Add the CUS threshold parameter, privileges and SoD rule",
          "description": "Parameter table field, security privilege/duty extensions, SoD rule between entering and approving invoices.",
          "req_ids": ["REQ-001", "REQ-006"], "estimate_hours": 10},
         {"task_id": "T-04", "kind": "Functional", "title": "FUNC: Give auditors read access to the workflow history",
          "description": "Role setup for auditors.", "req_ids": ["REQ-005"], "estimate_hours": 3},
     ]},
    {"story_id": "US-02", "title": "[Finance] Hold payments for vendors without bank details",
     "as_a": "AP Team Lead", "i_want": "vendors without an active bank account to be on payment hold",
     "so_that": "we never pay a vendor by mistake", "req_ids": ["REQ-004"],
     "acceptance_criteria": [
         {"ac_id": "AC-06", "req_ids": ["REQ-004"], "given": "an existing vendor without an active bank account",
          "when": "the vendor check batch job runs", "then": "On hold is set to Payment and invoices can still be registered"},
         {"ac_id": "AC-07", "req_ids": ["REQ-004"], "given": "a new vendor saved without a bank account",
          "when": "the vendor is saved", "then": "the vendor is placed on payment hold"},
     ],
     "tasks": [
         {"task_id": "T-05", "kind": "Technical", "title": "TECH: Build the CUSVendBankAccountCheck SysOperation batch job",
          "description": "Sets and releases On hold = Payment per legal entity, idempotent, logs every change.",
          "req_ids": ["REQ-004"], "estimate_hours": 16},
         {"task_id": "T-06", "kind": "Technical", "title": "TECH: Chain of Command validation on vendor insert/update and CUS log table",
          "description": "CoC on VendTable insert/update, CUSVendHoldLog table.", "req_ids": ["REQ-004"], "estimate_hours": 10},
         {"task_id": "T-07", "kind": "Technical", "title": "TECH: Write the TDD",
          "description": "Technical design document.", "req_ids": ["REQ-001", "REQ-003", "REQ-004", "REQ-006"], "estimate_hours": 6},
         {"task_id": "T-08", "kind": "Functional", "title": "FUNC: Schedule the batch job in USMF and DEMF and train the AP team",
          "description": "Batch recurrence and user training.", "req_ids": ["REQ-004"], "estimate_hours": 4},
     ]},
]}

In [ ]:
# Company standards (the same text that is stored in the knowledge base, notebook 02)
STANDARDS = {
    'STD-DEV-01': 'All custom objects use the prefix CUS and are placed in the model CUSExtensions. Customizations must use extensions and Chain of Command; overlayering and copying standard code are not allowed. Configurable values such as thresholds, days or amounts are stored in a parameter table exposed on a parameters form, never hard-coded. Every new menu item needs a privilege and a duty, delivered as extensions of existing roles. Business logic must have SysTest unit tests, and each change goes through pull-request code review in Azure DevOps.',
    'STD-DOC-01': 'A Functional Design Document (FDD) is required when a business process or configuration changes. It contains business context, as-is and to-be process, configuration and parameters, roles and security, exceptions, reports and data, assumptions and open points. A Technical Design Document (TDD) is required when code, integrations or reports are created. It contains solution overview, object list (extensions, classes, tables, security objects), logic, data model changes, error handling and logging, performance and batch considerations, and deployment steps. Mixed changes need both documents. Every section references the requirement IDs it covers.',
    'STD-ADO-01': "Each approved requirement is delivered through at least one User Story. Story titles use the format '[Module] short goal', and the description uses 'As a <role>, I want <goal> so that <benefit>'. Acceptance criteria are written as Given/When/Then and reference requirement IDs. Implementation work is split into child Tasks: functional tasks are prefixed 'FUNC:' (configuration, FDD, training) and technical tasks are prefixed 'TECH:' (development, TDD, unit tests). Tasks carry an estimate in hours in Remaining Work. Stories are tagged with the module, work type and request ID. Assignees are proposed and then confirmed by a person; tools must not assign work automatically.",
    'STD-TEST-01': 'Each requirement is covered by at least one test case, and test cases reference requirement IDs. Test types: Functional (business scenario in the UI), Technical (unit or component behaviour, such as a SysTest), Integration (interaction with workflow, batch, data entities or external systems) and Regression (standard processes that must still work). Threshold rules need boundary tests at, just below and just above the limit. Security requirements need a negative test proving that an unauthorized user is blocked.',
    'STD-COMM-01': 'Clarification emails are short, polite and written in business language without technical jargon. Questions are numbered, one topic per question, with at most eight questions per email. Each email restates the request in one sentence, asks for a reply date, and never promises a delivery date or solution before the requirement is approved. Emails are saved as drafts and sent only after a person reviews them.',
    'STD-CR-01': "A Change Request (CR) is required before any work that changes the agreed solution: new or changed functionality, customizations (X++ extensions), new or changed reports, integrations, data entities, workflows or fields, changes to the agreed scope, legal entities or business processes, and any configuration change outside the support contract. No CR is needed for support: defects in functionality that was delivered and accepted, errors caused by data or setup that are restored to the agreed design, how-to questions, access or security issues within the agreed roles, and standard Microsoft updates. When a request mixes both, the support part is handled as a ticket and the changed part as a CR. A CR uses the ERP Change/Additional Request Form: project name, change number, requested by, date of request, required phase, business priority, change name and technical complexity; the change description (business requirement and proposed solution) and the recommendation/best practice (customization required, affected areas, estimated effort) are written by the implementation partner; the business justification and the impact of not implementing the change are left for the customer's business to fill in; sign-off by the customer's functional lead, head of department and IT manager and by the partner's project manager. Work on a change starts only after the CR is signed.",
}

In [ ]:
def handoff_package(state: dict) -> str:
    """The structured package passed at every handoff (Coordination section of the planning document)."""
    package = {
        "request_id": state.get("request_id"),
        "source_text": state.get("source_text", ""),
        "requirements": state.get("requirements", []),
        "classification": state.get("classification"),
        "confirmed_answers": state.get("confirmed_answers", {}),
        "assumptions": state.get("assumptions", []),
        "retrieved_evidence": [
            {**e, "snippet": e["snippet"][:400]} for e in state.get("evidence", [])
        ],
        "approval_status": state.get("approval_status", {}),
        "immutable_facts": state.get("locked_facts", []),
    }
    return json.dumps(package, indent=2, ensure_ascii=False)
PLANNING_SYSTEM = """
You are the Planning Agent of a Dynamics 365 F&O delivery team.
Create Azure DevOps user stories from the approved specification.

Rules:
- Every requirement ID must be covered by at least one user story and at least one acceptance criterion.
- Story titles use the format "[Module] short goal". Acceptance criteria use Given/When/Then, IDs AC-01, AC-02, ...
- Split the work into tasks, IDs T-01, T-02, ...: functional tasks start with "FUNC:" (configuration,
  FDD, training) and technical tasks start with "TECH:" (X++ extensions, security objects, batch jobs,
  TDD, unit tests). Give each task an estimate in hours.
- The immutable facts are final: use their exact values and never change them.
"""

FDD_SYSTEM = """
You are the Functional Design Agent of a Dynamics 365 F&O delivery team.
Write the Functional Design Document (FDD) for the approved specification.

- Follow the company design document standard: business context, as-is and to-be process,
  configuration and parameters, roles and security, exceptions, reports and audit, assumptions.
- Section IDs FDD-1, FDD-2, ... Each section lists the requirement IDs it covers, and every requirement
  must appear in at least one section.
- Use only the immutable facts and confirmed answers. Cite evidence IDs in the text, e.g. [MS-WF-01].
"""

TDD_SYSTEM = """
You are the Technical Design Agent of a Dynamics 365 F&O delivery team.
Write the Technical Design Document (TDD) for the approved specification.

- Follow the company design and development standards: solution overview, object list (table and form
  extensions, Chain of Command classes, SysOperation batch classes, security privileges and duties),
  logic, data model changes, error handling and logging, performance, deployment.
- Extensions only (no overlayering), CUS prefix, configurable values in a parameter table.
- Section IDs TDD-1, TDD-2, ... Each section lists the requirement IDs it covers, and every requirement
  must appear in at least one section.
- Use only the immutable facts. Cite evidence IDs in the text, e.g. [DES-2025-031].
"""

TEST_SYSTEM = """
You are the Test Agent of a Dynamics 365 F&O delivery team.
Create test cases for the approved specification, IDs TC-01, TC-02, ...

- Types: Functional, Technical, Integration and Regression.
- Every requirement ID needs at least one test case.
- Threshold rules need boundary tests at, just below and just above the limit.
- Security rules need a negative test proving an unauthorized user is blocked.
- Add regression tests for standard processes that must keep working.
- Use the exact values from the immutable facts.
"""

VALIDATION_SYSTEM = """
You are the Validation Agent of a Dynamics 365 F&O delivery team.
Compare the artefacts with the immutable facts and with each other.

Report only real problems:
- a wrong value (amount, currency, legal entity, role, number of days, hold type, deadline),
- a contradiction between the design, the acceptance criteria and the tests,
- behaviour that is not in the approved requirements.
For each issue name the responsible agent: planning, functional_design, technical_design or tests.
If everything is consistent, return an empty list of issues.
"""

REVISION_INSTRUCTIONS = """
Revise your previous output. It failed validation.

Validation feedback:
{feedback}

Lessons from earlier attempts (reflexion memory):
{reflections}

Return the complete corrected artefact. Keep everything that was already correct.
"""

## The Design and Test Agents

- The **Functional Design Agent** writes the FDD sections required by the company standard: business context,
  as-is/to-be process, configuration, roles and security, exceptions, reports and audit, assumptions.
- The **Technical Design Agent** writes the TDD: object list (extensions, Chain of Command, SysOperation batch,
  security objects), logic, data model, error handling, performance and deployment.
- The **Test Agent** receives the acceptance criteria and the section headings of both designs, so the tests
  match what was designed.

Every section and every test lists the requirement IDs it covers. Notebook 06 uses these IDs to build the
traceability matrix.

In [ ]:
def plan_summary(plan: dict) -> str:
    lines = []
    for story in plan["user_stories"]:
        lines.append(f"{story['story_id']} {story['title']} ({', '.join(story['req_ids'])})")
        lines += [f"  {ac['ac_id']} ({', '.join(ac['req_ids'])}): Given {ac['given']}, when {ac['when']}, then {ac['then']}"
                  for ac in story["acceptance_criteria"]]
        lines += [f"  {task['task_id']} {task['title']}" for task in story["tasks"]]
    return "\n".join(lines)


@traceable(name="Planning Agent")
def run_planning_agent(state: dict) -> dict:
    user_prompt = f"APPROVED PACKAGE:\n{handoff_package(state)}\n\nWORK ITEM STANDARD:\n{STANDARDS['STD-ADO-01']}"
    return ask_structured(DeliveryPlan, PLANNING_SYSTEM, user_prompt).model_dump()


@traceable(name="Design Agent")
def run_design_agent(kind: str, state: dict) -> dict:
    """Functional (FDD) or Technical (TDD) Design Agent."""
    system_prompt = FDD_SYSTEM if kind == "FDD" else TDD_SYSTEM
    standards = STANDARDS["STD-DOC-01"] if kind == "FDD" else f"{STANDARDS['STD-DOC-01']}\n{STANDARDS['STD-DEV-01']}"
    user_prompt = (
        f"APPROVED PACKAGE:\n{handoff_package(state)}\n\n"
        f"DELIVERY PLAN:\n{plan_summary(state['plan'])}\n\n"
        f"COMPANY STANDARDS:\n{standards}\n\nWrite the {kind}."
    )
    document = ask_structured(DesignDocument, system_prompt, user_prompt).model_dump()
    document["doc_type"] = kind
    return document


@traceable(name="Test Agent")
def run_test_agent(state: dict) -> dict:
    design_sections = [
        f"{s['section_id']} {s['heading']} ({', '.join(s['req_ids'])})"
        for key in ("fdd", "tdd") if state.get(key) for s in state[key]["sections"]
    ]
    user_prompt = (
        f"APPROVED PACKAGE:\n{handoff_package(state)}\n\n"
        f"ACCEPTANCE CRITERIA AND TASKS:\n{plan_summary(state['plan'])}\n\n"
        "DESIGN SECTIONS:\n" + "\n".join(design_sections) + "\n\n"
        f"TEST STANDARD:\n{STANDARDS['STD-TEST-01']}"
    )
    return ask_structured(TestSuite, TEST_SYSTEM, user_prompt).model_dump()

## The Parallel Design Branch in LangGraph

Both design nodes start from `START`, so LangGraph runs them in the **same step**. The edge
`add_edge(["functional_design", "technical_design"], "test_agent")` makes the Test Agent wait until **both**
designs are ready.

If the classification needs only one document, the other node skips itself. The `timeline` field uses an
`operator.add` reducer, because both parallel nodes write to it at the same time.

In [ ]:
class DesignState(TypedDict, total=False):
    request_id: str
    source_text: str
    requirements: list
    classification: dict
    evidence: list
    approval_status: dict
    locked_facts: list
    plan: dict
    fdd: dict | None
    tdd: dict | None
    tests: dict
    timeline: Annotated[list, operator.add]


T0 = time.perf_counter()


def design_node(kind: str):
    key = kind.lower()

    def node(state: DesignState) -> dict:
        if kind not in state["classification"]["design_documents"]:
            return {key: None, "timeline": [f"{kind} skipped (not required)"]}
        start = time.perf_counter() - T0
        document = run_design_agent(kind, state)
        end = time.perf_counter() - T0
        print(f"[{kind}] {len(document['sections'])} sections ({start:.0f}s -> {end:.0f}s)")
        return {key: document, "timeline": [f"{kind} ran from {start:.0f}s to {end:.0f}s"]}

    return node


def test_node(state: DesignState) -> dict:
    start = time.perf_counter() - T0
    tests = run_test_agent(state)
    end = time.perf_counter() - T0
    print(f"[Tests] {len(tests['test_cases'])} test cases ({start:.0f}s -> {end:.0f}s)")
    return {"tests": tests, "timeline": [f"Tests ran from {start:.0f}s to {end:.0f}s"]}


graph = StateGraph(DesignState)
graph.add_node("functional_design", design_node("FDD"))
graph.add_node("technical_design", design_node("TDD"))
graph.add_node("test_agent", test_node)

graph.add_edge(START, "functional_design")
graph.add_edge(START, "technical_design")
graph.add_edge(["functional_design", "technical_design"], "test_agent")
graph.add_edge("test_agent", END)

design_branch = graph.compile()
print(design_branch.get_graph().draw_mermaid())

## Run the Branch

In [ ]:
T0 = time.perf_counter()
result = design_branch.invoke({
    "request_id": REQUEST_ID,
    "source_text": CUSTOMER_REQUEST,
    "requirements": APPROVED_REQUIREMENTS,
    "classification": CLASSIFICATION,
    "evidence": EVIDENCE,
    "approval_status": {"specification": "approved"},
    "locked_facts": LOCKED_FACTS,
    "plan": SAMPLE_PLAN,
})

show("TIMELINE", result["timeline"])

In [ ]:
for key in ("fdd", "tdd"):
    document = result[key]
    if document:
        print(f"\n==================== {document['title']} ====================")
        for section in document["sections"]:
            print(f"\n{section['section_id']} {section['heading']}   covers {', '.join(section['req_ids'])}")
            print(section["content"][:500])

In [ ]:
print("========== TEST CASES ==========\n")
for test in result["tests"]["test_cases"]:
    print(f"{test['test_id']} [{test['test_type']}] {test['title']}   covers {', '.join(test['req_ids'])}")
    print(f"      expected: {test['expected_result']}")

## Document Tool: Export to Word

The document tool writes the FDD, the TDD and the test cases (with a traceability matrix) as `.docx` files
using `python-docx`. The immutable facts are printed at the top of each design so reviewers can check them.

In [ ]:
COLUMN_OWNER = {
    "acceptance_criteria": "planning",
    "tasks": "planning",
    "fdd_sections": "functional_design",
    "tdd_sections": "technical_design",
    "tests": "tests",
}


def build_traceability(requirements: list, plan: dict, fdd: dict | None, tdd: dict | None, tests: dict):
    """Map every requirement ID to the acceptance criteria, tasks, design sections and tests that cover it."""
    matrix = {r["req_id"]: {column: [] for column in COLUMN_OWNER} for r in requirements}
    unknown_refs = []

    def add(req_ids, column, ref):
        for req_id in req_ids:
            if req_id in matrix:
                matrix[req_id][column].append(ref)
            else:
                unknown_refs.append((req_id, column, ref))

    for story in plan["user_stories"]:
        for ac in story["acceptance_criteria"]:
            add(ac["req_ids"], "acceptance_criteria", ac["ac_id"])
        for task in story["tasks"]:
            add(task["req_ids"], "tasks", task["task_id"])
    for design, column in ((fdd, "fdd_sections"), (tdd, "tdd_sections")):
        for section in (design or {}).get("sections", []):
            add(section["req_ids"], column, section["section_id"])
    for test in tests["test_cases"]:
        add(test["req_ids"], "tests", test["test_id"])

    return matrix, unknown_refs


def traceability_issues(matrix: dict, unknown_refs: list, design_documents: str) -> list:
    issues = []
    design_owner = "functional_design" if "FDD" in design_documents else "technical_design"

    for req_id, row in matrix.items():
        if not row["acceptance_criteria"]:
            issues.append({"severity": "Major", "responsible_agent": "planning", "req_ids": [req_id],
                           "description": f"{req_id} has no acceptance criterion.",
                           "fix_instruction": f"Add a Given/When/Then acceptance criterion for {req_id}."})
        if not row["fdd_sections"] and not row["tdd_sections"]:
            issues.append({"severity": "Major", "responsible_agent": design_owner, "req_ids": [req_id],
                           "description": f"{req_id} is not covered by any design section.",
                           "fix_instruction": f"Add or extend a design section that covers {req_id}."})
        if not row["tests"]:
            issues.append({"severity": "Major", "responsible_agent": "tests", "req_ids": [req_id],
                           "description": f"{req_id} has no test case.",
                           "fix_instruction": f"Add at least one test case for {req_id}."})

    for req_id, column, ref in unknown_refs:
        issues.append({"severity": "Major", "responsible_agent": COLUMN_OWNER[column], "req_ids": [req_id],
                       "description": f"{ref} references {req_id}, which is not an approved requirement.",
                       "fix_instruction": f"Only reference approved requirement IDs: {', '.join(matrix)}."})
    return issues


def print_matrix(matrix: dict) -> None:
    print(f"{'Requirement':<12}{'Accept. criteria':<22}{'FDD':<16}{'TDD':<16}{'Tests'}")
    for req_id, row in matrix.items():
        print(f"{req_id:<12}{', '.join(row['acceptance_criteria']) or '-':<22}"
              f"{', '.join(row['fdd_sections']) or '-':<16}{', '.join(row['tdd_sections']) or '-':<16}"
              f"{', '.join(row['tests']) or 'MISSING'}")
from datetime import date

import docx
from docx.shared import Pt


def _new_document(title: str, request_id: str, subtitle: str):
    document = docx.Document()
    document.styles["Normal"].font.size = Pt(10.5)
    document.add_heading(title, 0)
    document.add_paragraph(f"Request: {request_id}    |    {subtitle}    |    Generated: {date.today()}")
    return document


def export_design_document(design: dict, request_id: str, locked_facts: list, folder: Path) -> Path:
    """Document tool: write an FDD or TDD as a Word file."""
    document = _new_document(design["title"], request_id, design["doc_type"])
    document.add_heading("Overview", 1)
    document.add_paragraph(design["overview"])
    document.add_heading("Approved facts (immutable)", 1)
    for fact in locked_facts:
        document.add_paragraph(fact, style="List Bullet")
    for section in design["sections"]:
        document.add_heading(f"{section['section_id']}  {section['heading']}", 1)
        document.add_paragraph(section["content"])
        document.add_paragraph().add_run(f"Covers: {', '.join(section['req_ids'])}").italic = True
    if design["open_points"]:
        document.add_heading("Open points", 1)
        for point in design["open_points"]:
            document.add_paragraph(point, style="List Bullet")

    path = folder / f"{request_id}_{design['doc_type']}.docx"
    document.save(path)
    return path


def export_test_document(tests: dict, matrix: dict, request_id: str, folder: Path) -> Path:
    """Document tool: write the test cases and the traceability matrix as a Word file."""
    document = _new_document("Test Cases and Traceability", request_id, "Test plan")

    document.add_heading("Traceability matrix", 1)
    columns = ["acceptance_criteria", "fdd_sections", "tdd_sections", "tests"]
    table = document.add_table(rows=1, cols=len(columns) + 1)
    table.style = "Light Grid Accent 1"
    for cell, text in zip(table.rows[0].cells, ["Requirement", "Acceptance criteria", "FDD", "TDD", "Tests"]):
        cell.text = text
    for req_id, row in matrix.items():
        cells = table.add_row().cells
        cells[0].text = req_id
        for cell, column in zip(cells[1:], columns):
            cell.text = ", ".join(row[column]) or "-"

    document.add_heading("Test cases", 1)
    for test in tests["test_cases"]:
        document.add_heading(f"{test['test_id']}  {test['title']}", 2)
        document.add_paragraph(f"Type: {test['test_type']}    Requirements: {', '.join(test['req_ids'])}")
        document.add_paragraph(f"Preconditions: {test['preconditions']}")
        for step in test["steps"]:
            document.add_paragraph(step, style="List Number")
        document.add_paragraph(f"Expected result: {test['expected_result']}")

    path = folder / f"{request_id}_Test_Cases.docx"
    document.save(path)
    return path
output_folder = Path("output") / REQUEST_ID
output_folder.mkdir(parents=True, exist_ok=True)

matrix, _ = build_traceability(APPROVED_REQUIREMENTS, SAMPLE_PLAN, result["fdd"], result["tdd"], result["tests"])

exported = [export_design_document(result[key], REQUEST_ID, LOCKED_FACTS, output_folder)
            for key in ("fdd", "tdd") if result[key]]
exported.append(export_test_document(result["tests"], matrix, REQUEST_ID, output_folder))

for path in exported:
    print(f"Saved {path} ({path.stat().st_size:,} bytes)")

## Change Requests

A request that changes the agreed solution (new functionality, customization, reports, integrations, workflows,
fields or scope) needs a **Change Request** signed by the customer before the work starts. Support work does not:
defects in delivered functionality, data or setup fixes, questions and access issues. The rules come from the
company CR standard `STD-CR-01` in the knowledge base.

- The **CR Assessment Agent** decides *Support*, *Change request* or *Support and change request*, lists which
  parts are changes and which are support, and names the rule behind each reason.
- The **Change Request Agent** writes the partner's parts of the form: change name, business requirement,
  proposed solution, technical complexity, business priority, recommendation (customization required, affected
  areas, assumptions). The estimated effort comes from the delivery plan.
- The document tool writes the **ERP Change/Additional Request Form** in the company layout: header table,
  change description, the business justification and impact sections left empty for the customer's business,
  recommendation/best practice, and the sign-off table.

In [ ]:
CR_STANDARD = STANDARDS["STD-CR-01"]


class CRAssessment(BaseModel):
    request_type: Literal["Support", "Change request", "Support and change request"]
    cr_required: bool = Field(description="True when any part of the request needs a Change Request")
    reasons: list[str] = Field(description="Why, each reason naming the rule of the CR standard that applies")
    change_items: list[str] = Field(description="Parts that change the agreed solution (empty when none)")
    support_items: list[str] = Field(description="Parts that are support: defects, data or setup fixes, questions")


class ChangeRequestContent(BaseModel):
    change_name: str = Field(description="Short name of the change, e.g. 'Vendor invoice approval above 10,000 USD'")
    business_requirement: str = Field(description="What the customer needs and why, in business language (1-3 short paragraphs)")
    proposed_solution: str = Field(description="How it will be delivered in Dynamics 365 F&O, in business language")
    customization_required: bool
    technical_complexity: Literal["Low", "Medium", "High"]
    business_priority: Literal["Low", "Medium", "High"] = Field(description="From the customer's urgency and impact")
    recommendation: str = Field(description="Recommendation and best practice: standard features first, customization only where needed")
    affected_areas: list[str] = Field(description="Forms, journals, workflows, reports, entities or processes affected")
    assumptions: list[str]


CR_ASSESSMENT_SYSTEM = """
You are the Change Request Assessment Agent of a Dynamics 365 F&O partner.
Decide whether a customer request needs a Change Request (CR), using only the company CR standard you are given.
- A defect in functionality that already works as agreed, a data or setup problem, a how-to question or an
  access issue is support. New or changed functionality, customizations, reports, integrations, workflows,
  fields, processes or scope are changes.
- Name the rule of the standard that applies in each reason.
- When the request mixes both, list each part and set cr_required to true.
"""

CR_WRITER_SYSTEM = """
You are the Change Request Agent of a Dynamics 365 F&O implementation partner.
Write the partner's parts of the ERP Change/Additional Request Form for the customer to review and sign.
- Business language for the customer's managers; technical object names only in affected_areas.
- Use only facts from the request, the confirmed facts and the plan. Never invent amounts, dates or names.
- Recommend standard features first and say clearly when customization is required and why.
- Do not mention prices. The estimated effort is added from the plan.
"""


@traceable(name="CR Assessment Agent")
def assess_change_request(request_text: str, context: str = "") -> CRAssessment:
    user_prompt = f"Company CR standard:\n{CR_STANDARD}\n\nCustomer request:\n{request_text}"
    if context:
        user_prompt += f"\n\nContext:\n{context}"
    return ask_structured(CRAssessment, CR_ASSESSMENT_SYSTEM, user_prompt)


@traceable(name="Change Request Agent")
def write_change_request(request_text: str, change_items: list, context: str = "") -> ChangeRequestContent:
    user_prompt = (f"Company CR standard:\n{CR_STANDARD}\n\nCustomer request:\n{request_text}\n\n"
                   "Parts that are changes:\n" + "\n".join(f"- {item}" for item in change_items))
    if context:
        user_prompt += f"\n\nConfirmed facts and plan:\n{context}"
    return ask_structured(ChangeRequestContent, CR_WRITER_SYSTEM, user_prompt)


import math
import re
from datetime import date, datetime

import docx
from docx.shared import Pt

CR_PROJECT_NAME = os.getenv("CR_PROJECT_NAME", "Dynamics 365 F&O")
CR_SIGN_OFF = ["Functional Lead - Customer", "Head of Department - Customer", "IT Manager - Customer",
               "Project Manager - Partner"]


def form_date(received: str = "") -> str:
    """'2026-09-25T09:14:00Z' -> '25/09/2026', the form's date format (today when the date is unknown)."""
    try:
        return datetime.fromisoformat(received.replace("Z", "+00:00")).strftime("%d/%m/%Y")
    except ValueError:
        return date.today().strftime("%d/%m/%Y")


def cr_file_name(change_number) -> str:
    number = str(change_number)
    return re.sub(r"[^A-Za-z0-9_-]", "_", number if not number.isdigit() else f"CR-{number}") + "_Change_Request.docx"


def export_change_request(cr: dict, meta: dict, folder: Path) -> Path:
    """Document tool: the ERP Change/Additional Request Form (the company layout) as a Word file.
    meta: change_number, requested_by, date_of_request, phase, effort_hours."""
    document = docx.Document()
    document.styles["Normal"].font.name = "Arial"
    document.styles["Normal"].font.size = Pt(10)
    document.add_paragraph("ERP Change/Additional Request Form", style="Title")

    header = document.add_table(rows=4, cols=4)
    header.style = "Table Grid"
    values = [("Project Name", CR_PROJECT_NAME, "Change Number", meta["change_number"]),
              ("Requested By", meta["requested_by"], "Date of Request", meta["date_of_request"]),
              ("Required Phase of Project", meta["phase"], "Business Priority (High, Medium, Low)", cr["business_priority"]),
              ("Change Name", cr["change_name"], "Technical Complexity (High, Medium, Low)", cr["technical_complexity"])]
    for row, texts in zip(header.rows, values):
        for column, (cell, text) in enumerate(zip(row.cells, texts)):
            cell.paragraphs[0].add_run(str(text)).bold = column % 2 == 0  # labels in bold

    def section(title: str, blocks=()) -> None:
        """One form section: a bold title row and a body row. blocks: (kind, text) with kind heading/text/bullet."""
        document.add_paragraph()
        table = document.add_table(rows=2, cols=1)
        table.style = "Table Grid"
        table.rows[0].cells[0].paragraphs[0].add_run(title).bold = True
        body = table.rows[1].cells[0]
        for number, (kind, text) in enumerate(blocks):
            paragraph = body.paragraphs[0] if number == 0 else body.add_paragraph()
            run = paragraph.add_run(f"\u2022  {text}" if kind == "bullet" else text)
            run.bold = kind == "heading"

    def paragraphs(text: str) -> list:
        return [("text", line.strip()) for line in text.split("\n") if line.strip()]

    section("Change Description: (To be filled by the Partner)",
            [("heading", "Business Requirement"), *paragraphs(cr["business_requirement"]),
             ("heading", "Proposed Solution"), *paragraphs(cr["proposed_solution"])])
    section("Business Justification (Value): (Risk, productivity, efficiency, financial uplift) "
            "(To be filled by Customer Business)")
    section("Impact of not implementing the Change: (To be filled by Customer Business)")
    effort = meta["effort_hours"]
    section("Recommendation/Best Practice: (To be filled by the Partner)",
            [("heading", f"Customization Required: {'Yes' if cr['customization_required'] else 'No'}"),
             *paragraphs(cr["recommendation"]),
             ("heading", "Affected Areas"), *[("bullet", area) for area in cr["affected_areas"]],
             ("heading", "Estimated Effort"),
             ("text", f"{effort:g} hours (about {math.ceil(effort / 8)} working days), from the delivery plan"),
             *([("heading", "Assumptions")] + [("bullet", a) for a in cr["assumptions"]] if cr["assumptions"] else [])])

    document.add_paragraph()
    approval = document.add_table(rows=1 + len(CR_SIGN_OFF), cols=1)
    approval.style = "Table Grid"
    approval.rows[0].cells[0].paragraphs[0].add_run("For Approval:").bold = True
    for row, role in zip(approval.rows[1:], CR_SIGN_OFF):
        row.cells[0].text = f"Sign off by: ______________________ ({role})"

    folder.mkdir(parents=True, exist_ok=True)
    path = folder / cr_file_name(meta["change_number"])
    document.save(path)
    return path


CR_PARAGRAPH = {
    "Change request": "This request changes the agreed solution, so it needs a Change Request.",
    "Support and change request": "Part of this request changes the agreed solution, so that part needs a Change Request.",
}


def cr_reply_paragraph(cr: dict | None) -> str:
    """The acknowledgement paragraph that introduces the attached Change Request ("" when none is needed)."""
    if not cr:
        return ""
    return (f"**Change Request:**\n{CR_PARAGRAPH.get(cr['request_type'], CR_PARAGRAPH['Change request'])} "
            f"Please review the attached Change Request **{cr['number']}**, complete the business justification "
            "and impact sections, and return it signed. Work on the change starts once it is approved.\n\n")

### Does it Need a CR?

Three requests: CR-2026-001 (new approval workflow and vendor hold), a sales order that stopped confirming (a
defect), and an email that mixes a posting error with a new field on a report.

In [ ]:
CR_EXAMPLES = {
    REQUEST_ID: CUSTOMER_REQUEST,
    "Support email": ("Since this morning, sales orders in USMF cannot be confirmed. When we click 'Confirm sales "
                      "order' we get 'Update has been canceled'. It worked yesterday. The error log is attached."),
    "Mixed email": ("Posting free text invoices fails with a number sequence error since yesterday. Also, can you "
                    "add the customer's VAT number to the free text invoice report?"),
}

cr_assessments = {}
for ref, text in CR_EXAMPLES.items():
    assessment = assess_change_request(text)
    cr_assessments[ref] = assessment
    print(f"\n{ref}: {assessment.request_type} -> CR {'REQUIRED' if assessment.cr_required else 'not needed'}")
    for reason in assessment.reasons:
        print(f"   reason:  {reason}")
    for item in assessment.change_items:
        print(f"   change:  {item}")
    for item in assessment.support_items:
        print(f"   support: {item}")

### Write and Export the CR for CR-2026-001

The Change Request Agent works from the request, the immutable facts and the plan, so the CR states the
approved values. The change number is the request ID; the estimated effort is the sum of the plan's tasks.

In [ ]:
assessment = cr_assessments[REQUEST_ID]
if not assessment.cr_required:
    print("The assessment found no change, so no CR is written.")
else:
    context = "Immutable facts:\n" + "\n".join(LOCKED_FACTS) + "\n\nPlan:\n" + plan_summary(SAMPLE_PLAN)
    cr_content = write_change_request(CUSTOMER_REQUEST, assessment.change_items, context)
    effort = sum(task["estimate_hours"] for story in SAMPLE_PLAN["user_stories"] for task in story["tasks"])
    cr_path = export_change_request(cr_content.model_dump(), {
        "change_number": REQUEST_ID, "requested_by": "Rana Haddad (AP Team Lead, Cedar Retail Group)",
        "date_of_request": form_date("2026-09-25T09:14:00Z"), "phase": os.getenv("CR_PROJECT_PHASE", "Phase I"),
        "effort_hours": effort,
    }, output_folder)
    print(f"Saved {cr_path} ({cr_path.stat().st_size:,} bytes)\n")

    form = docx.Document(cr_path)  # read the form back
    for row in form.tables[0].rows:
        print(" | ".join(cell.text for cell in row.cells))
    for table in form.tables[1:5]:
        print(f"\n{table.rows[0].cells[0].text}\n{table.rows[1].cells[0].text or '(left for the customer)'}")

## SharePoint: File the Documents

Documents are filed in the support library on SharePoint as

**`<SHAREPOINT_ROOT_FOLDER>/<Customer>/<ID - Description>/<Document type>/<file>`**

with the document types `Change Request`, `Design`, `Test` and `Customer Files`. The site, library and root folder
come from `.env` (`SHAREPOINT_SITE_URL`, `SHAREPOINT_LIBRARY`, `SHAREPOINT_ROOT_FOLDER`); folder names are cleaned
of the characters SharePoint does not allow and kept short. An existing file with the same name gets a new
version (SharePoint keeps the history), so a signed CR can replace the draft.

Until SharePoint and Graph are configured, the same tree is written to `sharepoint_dry_run/` (dry-run). Live
access uses the app registration of notebook 03: delegated with `Sites.ReadWrite.All` (device-code sign-in), or
app-only with `GRAPH_CLIENT_SECRET` and the `Sites.Selected` permission granted on this site only. Only files
generated in `output/` or downloaded from a support email in `attachments/` can be uploaded.

In [ ]:
%%writefile sharepoint_client.py
"""SharePoint document filing through Microsoft Graph (live) or a local folder (dry-run).

Documents are filed as  <SHAREPOINT_ROOT_FOLDER>/<Customer>/<ID - Description>/<Document type>/<file>
in the library SHAREPOINT_LIBRARY of the site SHAREPOINT_SITE_URL.

Live mode needs SHAREPOINT_SITE_URL and Graph access (the app registration of notebook 03):
- Delegated (default): MailClient-style device-code sign-in with Sites.ReadWrite.All, cached in
  .graph_token_cache.json. Run SharePointClient().sign_in() once in the notebook.
- App-only: GRAPH_CLIENT_SECRET with the Sites.Selected permission, granted by an admin on this site only.
Only files generated in output/ or downloaded from a support email in attachments/ can be uploaded.
"""
import os
import re
import shutil
from pathlib import Path
from urllib.parse import quote, urlparse

import requests

GRAPH_URL = os.getenv("GRAPH_BASE_URL", "https://graph.microsoft.com/v1.0")
SCOPES = ["Sites.ReadWrite.All"]
TOKEN_CACHE = Path(".graph_token_cache.json")
DRY_RUN_DIR = Path("sharepoint_dry_run")
UPLOADABLE_DIRS = (Path("output"), Path("attachments"))
DOC_TYPES = ("Change Request", "Design", "Test", "Customer Files")
MAX_SIMPLE_UPLOAD_BYTES = 250 * 1024 * 1024


def safe_segment(text: str, max_length: int = 60) -> str:
    """One folder name SharePoint accepts: no " * : < > ? / \\ | # %, no leading/trailing dots or spaces."""
    text = re.sub(r'["*:<>?/\\|#%\x00-\x1f]', " ", str(text))
    text = re.sub(r"\s+", " ", text).strip(" .")
    if len(text) > max_length:  # cut at a word, and drop a dangling "and", "the", ... at the end
        text = text[:max_length].rsplit(" ", 1)[0]
        clause = text.lower().rfind(" and ")
        if clause > 20:  # end on a complete phrase: "Fix posting and add the customer's" -> "Fix posting"
            text = text[:clause]
        text = re.sub(r"(\s+(and|or|the|a|an|to|of|for|in|on|with|&|-))+$", "", text, flags=re.IGNORECASE)
    return text.rstrip(" .,-") or "Unnamed"


def ticket_folder(customer: str, item_id, description: str, doc_type: str | None = None) -> list:
    """['Customer', 'ID - Description', 'Document type'] below the root folder."""
    description = re.sub(rf"^{re.escape(customer)}\s*-\s*", "", description.strip(), flags=re.IGNORECASE)
    parts = [safe_segment(customer), safe_segment(f"{item_id} - {description}")]
    if doc_type:
        if doc_type not in DOC_TYPES:
            raise ValueError(f"Unknown document type {doc_type!r}: use one of {DOC_TYPES}")
        parts.append(doc_type)
    return parts


class SharePointClient:
    def __init__(self):
        self.site_url = os.getenv("SHAREPOINT_SITE_URL", "").rstrip("/")
        self.library = os.getenv("SHAREPOINT_LIBRARY", "Documents")
        self.root = [safe_segment(p) for p in os.getenv("SHAREPOINT_ROOT_FOLDER", "Support").split("/") if p.strip()]
        self.client_id = os.getenv("GRAPH_CLIENT_ID", "")
        self.tenant_id = os.getenv("GRAPH_TENANT_ID", "")
        self.secret = os.getenv("GRAPH_CLIENT_SECRET", "")
        self.static_token = os.getenv("GRAPH_ACCESS_TOKEN", "")
        self.live = bool(self.site_url and (self.static_token or (self.client_id and self.tenant_id)))
        self._drive_id = None

    @property
    def mode(self) -> str:
        return "live" if self.live else "dry-run"

    # ---------------- authentication (same app and token cache as the mail client) ----------------
    def _public_app(self):
        import msal

        cache = msal.SerializableTokenCache()
        if TOKEN_CACHE.exists():
            cache.deserialize(TOKEN_CACHE.read_text(encoding="utf-8"))
        app = msal.PublicClientApplication(
            self.client_id, authority=f"https://login.microsoftonline.com/{self.tenant_id}", token_cache=cache)
        return app, cache

    def sign_in(self) -> str:
        """Delegated sign-in with a device code for SharePoint access. Cached for later runs."""
        if not self.live or self.static_token or self.secret:
            return f"No interactive sign-in needed ({self.mode}{', app-only' if self.secret else ''})."
        app, cache = self._public_app()
        accounts = app.get_accounts()
        if accounts and app.acquire_token_silent(SCOPES, account=accounts[0]):
            return f"SharePoint access ready for {accounts[0]['username']}"
        flow = app.initiate_device_flow(scopes=SCOPES)
        if "user_code" not in flow:
            raise RuntimeError(f"Could not start the device sign-in: {flow.get('error_description', flow)}")
        print(flow["message"], flush=True)
        result = app.acquire_token_by_device_flow(flow)
        if "access_token" not in result:
            raise RuntimeError(f"Sign-in failed: {result.get('error_description', result)}")
        if cache.has_state_changed:
            TOKEN_CACHE.write_text(cache.serialize(), encoding="utf-8")
        return f"Signed in as {result.get('id_token_claims', {}).get('preferred_username', 'unknown user')}"

    def _token(self) -> str:
        if self.static_token:
            return self.static_token
        import msal

        if self.secret:
            app = msal.ConfidentialClientApplication(
                self.client_id, client_credential=self.secret,
                authority=f"https://login.microsoftonline.com/{self.tenant_id}")
            result = app.acquire_token_for_client(scopes=["https://graph.microsoft.com/.default"])
        else:
            app, cache = self._public_app()
            accounts = app.get_accounts()
            result = app.acquire_token_silent(SCOPES, account=accounts[0]) if accounts else None
            if not result:
                raise PermissionError("No SharePoint access yet: run SharePointClient().sign_in() in the notebook.")
            if cache.has_state_changed:
                TOKEN_CACHE.write_text(cache.serialize(), encoding="utf-8")
        if "access_token" not in result:
            raise PermissionError(f"Could not get a Graph token: {result.get('error_description', result)}")
        return result["access_token"]

    def _graph(self, method: str, path: str, ok=(), **kwargs) -> requests.Response:
        response = requests.request(method, f"{GRAPH_URL}{path}", timeout=120,
                                    headers={"Authorization": f"Bearer {self._token()}", **kwargs.pop("headers", {})},
                                    **kwargs)
        if not response.ok and response.status_code not in ok:
            raise requests.HTTPError(f"Graph {method} {path} -> {response.status_code}: {response.text[:400]}",
                                     response=response)
        return response

    # ---------------- site, library and folders ----------------
    def drive_id(self) -> str:
        """The document library of the site (matched by name, e.g. 'Documents' = 'Shared Documents')."""
        if self._drive_id:
            return self._drive_id
        url = urlparse(self.site_url)
        site = self._graph("GET", f"/sites/{url.hostname}:{quote(url.path)}").json()
        drives = self._graph("GET", f"/sites/{site['id']}/drives").json()["value"]
        wanted = self.library.lower()
        for drive in drives:
            names = {drive.get("name", "").lower(), drive.get("webUrl", "").rstrip("/").rsplit("/", 1)[-1].replace("%20", " ").lower()}
            if wanted in names or (wanted == "documents" and "shared documents" in names):
                self._drive_id = drive["id"]
                return self._drive_id
        raise ValueError(f"No library named {self.library!r} on {self.site_url}: {[d.get('name') for d in drives]}")

    @staticmethod
    def _path(parts: list) -> str:
        return "/".join(quote(part, safe="") for part in parts)

    def ensure_folder(self, parts: list) -> dict:
        """Create the folders below the root that do not exist yet; return the last folder."""
        full = self.root + [safe_segment(p, 120) for p in parts]
        if not self.live:
            folder = DRY_RUN_DIR.joinpath(*full)
            folder.mkdir(parents=True, exist_ok=True)
            return {"path": "/".join(full), "web_url": folder.resolve().as_uri()}

        drive = self.drive_id()
        item = None
        for depth in range(1, len(full) + 1):
            found = self._graph("GET", f"/drives/{drive}/root:/{self._path(full[:depth])}", ok=(404,))
            if found.status_code != 404:
                item = found.json()
                continue
            parent = f"/drives/{drive}/root" + (f":/{self._path(full[:depth - 1])}:" if depth > 1 else "")
            created = self._graph("POST", f"{parent}/children", ok=(409,), json={
                "name": full[depth - 1], "folder": {}, "@microsoft.graph.conflictBehavior": "fail"})
            item = created.json() if created.status_code != 409 else \
                self._graph("GET", f"/drives/{drive}/root:/{self._path(full[:depth])}").json()  # created meanwhile
        return {"path": "/".join(full), "web_url": item["webUrl"]}

    @staticmethod
    def _uploadable(path: str) -> Path:
        file = Path(path).resolve()
        if not any(file.is_relative_to(folder.resolve()) for folder in UPLOADABLE_DIRS):
            raise PermissionError(f"Only files from output/ or attachments/ can be uploaded: {path}")
        if not file.is_file():
            raise FileNotFoundError(f"File not found: {path}")
        if file.stat().st_size > MAX_SIMPLE_UPLOAD_BYTES:
            raise ValueError(f"{file.name} is larger than 250 MB.")
        return file

    def upload(self, parts: list, path: str) -> dict:
        """Upload one file into the folder (a new version when the file already exists)."""
        file = self._uploadable(path)
        folder = self.ensure_folder(parts)
        if not self.live:
            target = DRY_RUN_DIR.joinpath(*folder["path"].split("/"), file.name)
            shutil.copy2(file, target)
            return {"name": file.name, "size": file.stat().st_size, "web_url": target.resolve().as_uri()}
        item = self._graph(
            "PUT", f"/drives/{self.drive_id()}/root:/{self._path(folder['path'].split('/') + [file.name])}:/content",
            data=file.read_bytes(), headers={"Content-Type": "application/octet-stream"}).json()
        return {"name": file.name, "size": item.get("size"), "web_url": item["webUrl"]}

    def file_documents(self, customer: str, item_id, description: str, doc_type: str, paths: list) -> dict:
        """File documents as <root>/<Customer>/<ID - Description>/<Document type>/ and report every file."""
        parts = ticket_folder(customer, item_id, description, doc_type)  # validates the type before creating folders
        ticket = self.ensure_folder(parts[:-1])
        files = []
        for path in paths:
            try:
                files.append({**self.upload(parts, path), "status": "uploaded"})
            except Exception as error:  # report every failed file instead of hiding it
                files.append({"name": Path(path).name, "status": f"failed: {error}"})
        return {"mode": self.mode, "ticket_folder": ticket, "folder": "/".join(self.root + parts), "files": files}

In [ ]:
import importlib

import sharepoint_client

importlib.reload(sharepoint_client)
sharepoint = sharepoint_client.SharePointClient()
print("SharePoint mode:", sharepoint.mode)
print(sharepoint.sign_in())  # live + delegated: device-code sign-in the first time, cached afterwards

In [ ]:
CUSTOMER = "Cedar Retail Group"
# The email subject keeps the folder name stable from one run to the next
DESCRIPTION = CUSTOMER_REQUEST.strip().splitlines()[0].removeprefix("Subject:").strip()

filings = [
    sharepoint.file_documents(CUSTOMER, REQUEST_ID, DESCRIPTION, "Design",
                              [str(p) for p in exported if p.stem.endswith(("_FDD", "_TDD"))]),
    sharepoint.file_documents(CUSTOMER, REQUEST_ID, DESCRIPTION, "Test",
                              [str(p) for p in exported if p.stem.endswith("_Test_Cases")]),
]
if assessment.cr_required:
    filings.append(sharepoint.file_documents(CUSTOMER, REQUEST_ID, DESCRIPTION, "Change Request", [str(cr_path)]))

print(f"Mode: {sharepoint.mode}\nTicket folder: {filings[0]['ticket_folder']['path']}\n")
for filing in filings:
    for file in filing["files"]:
        print(f"{filing['folder']}/{file['name']}: {file['status']}")

## Observation

The classification of CR-2026-001 is *Mixed*, so both designs were needed. LangGraph started the Functional
and Technical Design Agents in the same step, and the timeline shows the two runs overlapping. (Whether Ollama
processes both requests at the same time depends on its `OLLAMA_NUM_PARALLEL` setting; the graph itself does
not wait for one design before starting the other.)

The FDD described the process and configuration: the workflow condition on the invoice total, the Finance Manager
step with CFO escalation, the payment hold and auditor access. The TDD described the objects the standard asks
for: a CUS parameter, security privileges and duties, the SysOperation batch job and the Chain of Command
validation on the vendor. Both reused the approved designs from the knowledge base as evidence.

The Test Agent ran only after both designs were ready and produced functional, technical, integration and
regression tests, including boundary tests around 10,000 USD and a negative test for self-approval.

Every section and test carries requirement IDs, which is what makes the validation in notebook 06 possible.

The CR Assessment Agent separated changes from support: CR-2026-001 needs a CR, the sales order that stopped
confirming is a defect handled as support, and the mixed email was split into a support part and a CR part.
The CR for CR-2026-001 follows the company form, states the approved facts, takes its effort from the plan,
and leaves the business justification and impact sections for the customer to complete before signing.